# SQuAD Context → New Quiz Question Dataset

This notebook generates exactly one new English quiz question for each filtered SQuAD context using the Gemini API. It saves progress to Google Drive and creates Unsloth-compatible train/validation/test JSONL files.

**Security:** Enter the Gemini API key only through the hidden prompt. Never save it in the notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Set paths

Change `DRIVE_PROJECT_DIR` only if the folder name differs. The expected input is `data/squad_contexts_1200.jsonl`.


In [ ]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/context-quiz-generator')
DATA_DIR = DRIVE_PROJECT_DIR / 'data'
INPUT_PATH = DATA_DIR / 'squad_contexts_1200.jsonl'
PROGRESS_PATH = DATA_DIR / 'generated_questions_progress.jsonl'
FINAL_DIR = DATA_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', DRIVE_PROJECT_DIR)
print('Input exists:', INPUT_PATH.exists(), INPUT_PATH)


In [ ]:
!pip -q install google-genai


## 2. Connect to Gemini securely

The API key is hidden while typing and is kept only in the current runtime memory.


In [ ]:
import getpass
from google import genai

GEMINI_API_KEY = getpass.getpass('Enter Gemini API key: ')
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.1-flash-lite-preview'
print('Gemini client ready:', MODEL_NAME)


## 3. Load contexts and test 5 samples

Run the five-sample test first. Inspect the outputs before generating the full dataset.


In [ ]:
import json, re, time

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

rows = read_jsonl(INPUT_PATH)
print('Contexts:', len(rows))


In [ ]:
PROMPT_TEMPLATE = """Read the passage below and generate exactly one meaningful quiz question that can be answered using only the passage.

Requirements:
- Ask about an important fact from the passage.
- Generate exactly one question.
- Output only the question.
- Do not include the answer, choices, explanation, or any prefix.

Passage:
{context}
"""

def clean_question(text):
    text = ' '.join(text.strip().split())
    text = re.sub(r'^(Question|Quiz question)\s*:\s*', '', text, flags=re.I)
    first = text.split('?')[0].strip() + '?' if '?' in text else text
    return first

def generate_question(context):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=PROMPT_TEMPLATE.format(context=context),
    )
    return clean_question(response.text)

for row in rows[:5]:
    q = generate_question(row['context'])
    print('TITLE:', row['title'])
    print('QUESTION:', q, '\n')
    time.sleep(1)


## 4. Generate all questions with resumable progress

The notebook appends each successful result immediately. If Colab disconnects, rerun the notebook and this cell will resume from the next unfinished context. Free-tier rate limits may require a longer delay or multiple sessions.


In [ ]:
def is_valid_question(question):
    words = question.split()
    return (
        question.endswith('?')
        and question.count('?') == 1
        and 5 <= len(words) <= 40
    )

completed = {}
if PROGRESS_PATH.exists():
    for row in read_jsonl(PROGRESS_PATH):
        completed[row['context_id']] = row
print('Already completed:', len(completed))

MAX_RETRIES = 3
DELAY_SECONDS = 4

with open(PROGRESS_PATH, 'a', encoding='utf-8') as out:
    for index, row in enumerate(rows, start=1):
        if row['context_id'] in completed:
            continue
        for attempt in range(MAX_RETRIES):
            try:
                question = generate_question(row['context'])
                result = {**row, 'generated_question': question, 'valid_format': is_valid_question(question)}
                out.write(json.dumps(result, ensure_ascii=False) + '\n')
                out.flush()
                completed[row['context_id']] = result
                print(f"[{index}/{len(rows)}] {question}")
                break
            except Exception as error:
                wait = DELAY_SECONDS * (attempt + 1)
                print(f"[{index}] attempt {attempt + 1} failed: {error}; wait {wait}s")
                time.sleep(wait)
        time.sleep(DELAY_SECONDS)

print('Completed:', len(completed))


## 5. Validate and create Unsloth chat-format splits

Only format-valid questions are included. Splits are deterministic and keep one unique context per sample.


In [ ]:
import random

INSTRUCTION = 'Read the passage and generate exactly one meaningful quiz question that can be answered using only the passage. Output only the question.'
all_generated = read_jsonl(PROGRESS_PATH)
valid = [r for r in all_generated if is_valid_question(r['generated_question'])]
invalid = [r for r in all_generated if not is_valid_question(r['generated_question'])]
print('Generated:', len(all_generated), 'Valid:', len(valid), 'Invalid:', len(invalid))

chat_rows = []
for row in valid:
    chat_rows.append({
        'context_id': row['context_id'],
        'messages': [
            {'role': 'user', 'content': f"{INSTRUCTION}\n\nPassage:\n{row['context']}"},
            {'role': 'assistant', 'content': row['generated_question']},
        ],
    })

random.Random(3407).shuffle(chat_rows)
train_end = int(len(chat_rows) * 0.8)
val_end = train_end + int(len(chat_rows) * 0.1)
splits = {'train': chat_rows[:train_end], 'validation': chat_rows[train_end:val_end], 'test': chat_rows[val_end:]}

for name, split in splits.items():
    path = FINAL_DIR / f'{name}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for row in split:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(name, len(split), path)


## 6. Manual quality-check sample

Review at least 100 random examples before training. Check relevance, answerability from the passage, naturalness, and whether only one question is generated.


In [ ]:
import random
for row in random.Random(3407).sample(valid, min(20, len(valid))):
    print('TITLE:', row['title'])
    print('CONTEXT:', row['context'][:300] + '...')
    print('QUESTION:', row['generated_question'])
    print('-' * 80)
